# 025 — Training the deterministic architectures with alpha sweep

The base `020` notebook trains each deterministic architecture once, with the default loss weight `ALPHA = 0.16`.  
This notebook trains a chosen subset of deterministic models across a grid of alpha values `0.16, 0.50, 0.84`.  
The loss is:  
```
combined_loss = alpha * Charbonnier  +  (1 - alpha) * (1 - MS-SSIM)
```

Every run goes to `models/alpha_sweep/alpha_<a>/<arch>/`.

Imports and a GPU sanity check.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset split
**Split used project-wide:**  
- Real **artworks** are grouped and kept entirely within one fold, exactly as a plain grouped split would do, so no painting leaks across train/val/test.  
- The **mockup** groups (listed in `settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from. They are split at the individual-pair level, with only `settings.MOCKUP_TEST_RATIO` (default 5%) held out for test. 


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

print(f"Train: {len(train_pairs)} patches")
print(f"Val:   {len(val_pairs)} patches")

## 2. The alpha grid


| label | alpha | (1 - alpha) | favours |
|---|---|---|---|
| `ssim_heavy` | 0.16 | 0.84 | structure — the Zhao et al. (2016) `Mix` ratio, `settings.LOSS_ALPHA` |
| `balanced` | 0.50 | 0.50 | neither |
| `l1_heavy` | 0.84 | 0.16 | pixel fidelity |


In [ ]:
ALPHAS = [0.16, 0.50, 0.84]  # combined_loss Charbonnier weight; 0.16 == settings.LOSS_ALPHA

assert settings.LOSS_ALPHA in ALPHAS, (
    f"settings.LOSS_ALPHA ({settings.LOSS_ALPHA}) is not in the grid — add it "
    "so the sweep has an anchor comparable to 020's runs."
)
for a in ALPHAS:
    print(f"alpha={a:.2f}  charbonnier={a:.2f}  ms_ssim={1 - a:.2f}")

## 3. Train

Each `(arch, alpha)` pair trains in its own subprocess.  
A pair whose checkpoint already exists is skipped, so the sweep
is resumable.

In [ ]:
import json
import subprocess

# Comment out any architecture to skip it. Values are builder kwargs.
ARCHS = {
    "unet": {},
    "resunet": {},
    "attention_unet": {},
    # "unet_residual": {},
    # "unet_v2": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
    # "unet_restormer": dict(num_heads=8, ffn_expansion_factor=2),
    # "unet_dilated": {},
    # "unet_v2_dilated": dict(use_strided_conv=True, use_upsample_conv=True, dropout_rate=0.2),
}

EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
SWEEP_DIR = settings.MODELS_DIR / "alpha_sweep"
SWEEP_LOG_DIR = settings.LOGS_DIR / "alpha_sweep"


def run_dirs(alpha: float) -> tuple[Path, Path]:
    tag = f"alpha_{alpha:.2f}"
    return SWEEP_DIR / tag, SWEEP_LOG_DIR / tag


histories: dict = {}

for alpha in ALPHAS:
    model_dir, log_dir = run_dirs(alpha)
    for arch, kwargs in ARCHS.items():
        ckpt = model_dir / arch / "best_model.keras"
        hist_path = model_dir / arch / "history.json"
        if ckpt.exists():
            print(f"[skip] {arch} alpha={alpha:.2f} -- exists ({ckpt})")
            histories[(arch, alpha)] = json.loads(hist_path.read_text())
            continue

        cmd = [
            sys.executable, "-m", "scripts.train_single",
            "--arch", arch,
            "--epochs", str(EPOCHS),
            "--model-dir", str(model_dir),
            "--log-dir", str(log_dir),
            "--kwargs", json.dumps(kwargs),
            "--loss-alpha", str(alpha),
        ]
        subprocess.run(cmd, cwd=project_root, check=True)

        histories[(arch, alpha)] = json.loads(hist_path.read_text())
        best = min(histories[(arch, alpha)]["val_loss"])
        print(f"\nBest val_loss ({arch}, alpha={alpha:.2f}): {best:.4f}")

## 4. Training curves

In [ ]:
for (arch, alpha), history in histories.items():
    plot_training_curves(history, title=f"{arch} -- alpha={alpha:.2f}")
    plt.show()

## 5. Summary

In [ ]:
print(f"{'arch':<18}{'alpha':<8}{'ckpt':<9}{'epochs':<8}{'val_loss':<11}{'val_mae':<11}")
print("-" * 71)
for alpha in ALPHAS:
    model_dir, _ = run_dirs(alpha)
    for arch in ARCHS:
        h = histories.get((arch, alpha))
        if h is None:
            print(f"{arch:<18}{alpha:<8.2f}{'MISSING':<9}")
            continue
        print(f"{arch:<18}{alpha:<8.2f}{'found':<9}{len(h['val_loss']):<8}"
              f"{min(h['val_loss']):<11.4f}{min(h['val_mae']):<11.4f}")

print(f"\nCheckpoints under models/alpha_sweep/alpha_<a>/<arch>/ "
      f"(020's models/deterministic/ untouched).")
print(f"Only alpha == settings.LOSS_ALPHA ({settings.LOSS_ALPHA}) is "
      f"loss-comparable to 020's runs.")